# 3.6 Audit Trail of Agent Actions: Provenance Graph vs. Text Log

*Formerly notebook 24.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): 1,332 questions, 7,144 calls through the
> Batch API, $3.40. Task 3.6. They are reported in `RESULTS-REPORT.md` and the v2 paper.

**Presentation map:** Task 3.6 (audit trail of agent actions; deck markers G23, G29, R4), Group 3 (governance graph).

An agent that acts on a user's behalf must be auditable: *which input led it to call that tool? which runs touched
this account? when did it first read text from a third party?* The tutorial proposes to record each action with its
inputs, sources and outcome as a **provenance graph** (W3C PROV-O: `Activity`, `Entity`, `Agent`; `used`,
`wasGeneratedBy`, `wasInformedBy`, `wasAssociatedWith`) and to answer audit questions with graph queries. The low-cost
alternative is the ordinary text log, read by an LLM or searched by an agent with grep. No published work compares the
two on the same runs (`PUBLISHED_RESULTS.md` Part C, 3.6: PROV-AGENT is evaluated qualitatively only).

## Protocol (frozen before the test run; PLAN §5, §5b)

**Research question.** Can audit questions about an agent's actions be answered more accurately from a provenance
graph of those actions than from a plain text log of the same events, and does the answer depend on the log length?

**Hypothesis.** Graph queries are at least as accurate as both log baselines on single-run lookups and more accurate on
counting and listing questions across runs; the advantage grows from 1-run to 200-run logs, where an LLM reading the
whole log is known to degrade (TRAIL, Oolong).

**Data.** The recorded traces of three earlier notebooks, read from their local caches (no new agent runs):
nb 13 InjecAgent (6,324 runs, defence `none` / `provenance`), nb 18 Aura helpdesk agent (271 runs, template and
independent questions), nb 19 MetaQA graph agent (1,800 runs, KB and extracted graph). Every run becomes the same list
of events: user request, agent steps (LLM calls), tool calls with their arguments, tool outputs with a trust label,
final answer. Tool outputs, requests and answers are cut to 400 characters *before* both renderings, so the two
records hold exactly the same information (checked in code: the event list is recovered unchanged from the text log
and from the graph).

**Audit logs.** Logs of **1, 10, 50 and 200 runs** sampled per system (seed 24); runs do not repeat within one
(system, length) cell, except that Aura has only 271 runs, so its two 200-run logs share 129 runs.

**Questions.** 1,332 template questions (4 lengths × 3 systems × 111), gold answers computed from the events:
(1) how many runs called tool T [with argument a = v]; (2) which runs called T with a = v (list); (3) how many tool
calls run R made; (4) which entities the step that issued a given call used (list); (5) which tool outputs contain a
value V that the final answer mentions (list); (6) the first step of run R that used untrusted content; (7) which runs
issued a tool call at or after their first untrusted input (list). Templates 6–7 only for InjecAgent and Aura (MetaQA
has no untrusted content).

**Conditions** (same model `gpt-6-luna`, no reasoning, temperature 0; the same description of the record's semantics
and answer format in every prompt):
- **(a) Full log:** one LLM call reads the whole text log and answers. Nothing is truncated to make a log fit: each
  log of 50+ runs is first sent once with a 16-token limit; if the API rejects it as too long, condition (a) is
  reported as *not runnable* on that log's questions, and the graph − full-log comparison uses the others.
- **(b) Search agent:** an agent with `grep` and `read_lines` tools over the text log; at most 3 tool-using steps and a
  4th answering step; each step is one `llm.run` (one Batch job).
- **(c) PROV graph:** the LLM receives the graph schema and writes one SPARQL query; it is run with rdflib on the
  log's PROV graph; one repair call if the query raises an error; a final call answers from the query result only.

**Primary metric.** Accuracy (exact match: the count, the step number, or the exact set of ids), pooled over all
questions. Primary comparisons: (c) − (b) and (c) − (a), paired 95% bootstrap over questions.
**Secondary.** Completeness (recall of the gold set) and precision on list questions; accuracy by log length and by
template; tokens per answer (all calls of a condition); query-error rate of (c).

**Scoring** is deterministic (no LLM judge): ids are compared after stripping prefixes/IRIs and case; counts and step
numbers by their integer value; `none` / `[]` is the empty answer.

**n.** 1,332 questions (333 per log length). Test run: the first 25 questions per log length for 1/10/50 runs and the
first 10 for 200 runs (85).

**Threats.** (i) The gold answers are computed from the same events as the graph and the log, so the benchmark measures
retrieval from the record, not whether the record is faithful to what happened. (ii) The questions are template
questions whose vocabulary matches the graph schema (`used`, `trust`, step index); the same definitions are given to
all conditions in words. (iii) Content is truncated at 400 characters; questions about text beyond that are not asked.
(iv) The trust labels are ours: every InjecAgent tool output (third-party content) and every Aura output containing a
ticket or comment record (user-written text) is `untrusted`. (v) One model, one prompt per condition, one run.

**Protocol changes after the test run:**
1. Implementation fix, not a protocol change: in the first test attempt
   several correct SPARQL queries over 50-run graphs ran for minutes in rdflib (its join order makes cross products), so
   a greedy join-order planner (result-preserving; checked on the test queries: identical results on all 68 queries
   that finish without it) and a 60 s limit per query were added before the test run reported here. No prompt,
   template or threshold was changed.
2. **Added 200-run logs** (lengths 1/10/50 → 1/10/50/200; 999 → 1,332 questions; two 200-run logs per system), decided
   before the full run and with no full-run result seen. Reason: test accuracy was 84–100% at 50 runs in all three
   conditions, a ceiling risk that would hide a length effect, and `PUBLISHED_RESULTS.md` (3.6) suggests 10/50/200.
   The questions of lengths 1/10/50 are unchanged (generated first, with the same seed). The context check for
   condition (a) was added with it. The 200-run test used its first 10 questions only.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, gzip, random, time, math
from collections import Counter, defaultdict
import numpy as np
import rdflib
from rdflib import Graph, Namespace, Literal, URIRef, BNode
from rdflib.namespace import RDF, XSD
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/audit_trail"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))
# test mode: the first 25 questions per log length (10 for the 200-run logs), in the same fixed order
N_PER_LEN = {1: 25, 10: 25, 50: 25, 200: 10}
LENGTHS = (1, 10, 50, 200)
N_LOGS = {1: 37, 10: 12, 50: 4, 200: 2}   # logs per (system, length)
Q_PER_CELL = 111                     # questions per (system, length): 4 x 3 x 111 = 1,332
CUT = 400                            # characters kept of every request, tool output and answer (both records)
print("MODE =", MODE, "| rdflib", rdflib.__version__)

MODE = batch | rdflib 7.6.0


## The traces: one event list per run

The input/output logs (`<cache>/io/<stage>.jsonl.gz`) store every request body with the model's response. The last
request of a run holds its whole history (user request, assistant tool calls, tool outputs); its response is the final
step (an answer, or a tool call that ended the run, e.g. an attacker tool call in InjecAgent). In InjecAgent the first
tool call and its output are part of the benchmark case (pre-filled), so they appear as step 1.

In [2]:
def clip(s, n=CUT):
    s = re.sub(r"\s+", " ", s or "").strip()
    return s if len(s) <= n else s[:n] + " …"

def canon_args(a):
    try:
        return json.dumps(json.loads(a), ensure_ascii=False, separators=(",", ":"))
    except Exception:
        return re.sub(r"\s+", " ", a or "")

def io_rows(path):
    with gzip.open(path, "rt") as f:
        for line in f:
            yield json.loads(line)

def last_requests(paths):
    last = {}
    for p in paths:
        if os.path.exists(p):
            for r in io_rows(p):
                last[r["key"]] = r                     # later steps overwrite earlier ones
    return last

def to_run(rid, system, agent, msgs, final, trust):
    user = next(m for m in msgs if m["role"] == "user")["content"]
    run = {"id": rid, "system": system, "agent": agent, "steps": [], "answer": None,
           "request": {"id": f"{rid}.req", "text": clip(user), "trust": "trusted"}}
    pending, by_tcid, k = [run["request"]["id"]], {}, 0
    def add_step(calls):
        nonlocal k, pending
        k += 1
        step = {"k": k, "used": sorted(pending), "calls": []}
        for j, (name, args, tcid) in enumerate(calls, 1):
            c = {"id": f"{rid}.c{k}.{j}", "tool": name, "args": canon_args(args), "out": None}
            step["calls"].append(c); by_tcid[tcid] = c
        run["steps"].append(step); pending = []
        return step
    for m in msgs:
        if m["role"] == "assistant" and m.get("tool_calls"):
            add_step([(t["function"]["name"], t["function"]["arguments"], t["id"]) for t in m["tool_calls"]])
        elif m["role"] == "tool":
            c = by_tcid[m["tool_call_id"]]
            c["out"] = {"id": c["id"].replace(".c", ".o", 1), "text": clip(m["content"]), "trust": trust(m["content"])}
            pending.append(c["out"]["id"])
    step = add_step([(t["name"], t["arguments"], f"final{j}") for j, t in enumerate(final.get("tool_calls") or [])])
    if not final.get("tool_calls") and "error" not in final:
        run["answer"] = {"id": f"{rid}.ans", "step": step["k"], "text": clip(final.get("text", ""))}
    return run

RUNS = {}
inj = "data/benchmarks/injecagent/cache/io"
RUNS["injecagent"] = [to_run("inj-" + k.replace("|", "-"), "injecagent", "injecagent-" + k.split("|")[0],
                             r["request"]["messages"], r["response"], lambda text: "untrusted")
                      for k, r in last_requests([f"{inj}/step1.jsonl.gz", f"{inj}/step2.jsonl.gz"]).items()]
aura_trust = lambda text: "untrusted" if re.search(r"\b(tkt|cmt)_\d+", text) else "trusted"
RUNS["aura"] = []
for pref, ag in (("agent_step", "aura-template"), ("indep_agent_step", "aura-independent")):
    last = last_requests([f"data/cache_18/io/{pref}{i}.jsonl.gz" for i in range(1, 11)])
    RUNS["aura"] += [to_run(f"aura-{k}", "aura", ag, r["request"]["messages"], r["response"], aura_trust)
                     for k, r in last.items()]
last = last_requests([f"data/benchmarks/metaqa/cache_19/io/agent_step{i}.jsonl.gz" for i in range(1, 10)])
RUNS["metaqa"] = [to_run("mqa-" + k.replace("|", "-"), "metaqa", "metaqa-" + k.split("|")[0],
                         r["request"]["messages"], r["response"], lambda text: "trusted") for k, r in last.items()]
for s, rs in RUNS.items():
    calls = [c for r in rs for st in r["steps"] for c in st["calls"]]
    print(f"{s:<11}{len(rs):>6,} runs | {np.mean([len(r['steps']) for r in rs]):.1f} steps, "
          f"{len(calls) / len(rs):.1f} tool calls per run | tools: {len({c['tool'] for c in calls})} | "
          f"untrusted outputs {np.mean([c['out']['trust'] == 'untrusted' for c in calls if c['out']]):.0%} | "
          f"runs without final answer {sum(r['answer'] is None for r in rs)}")

injecagent  6,324 runs | 2.0 steps, 1.1 tool calls per run | tools: 32 | untrusted outputs 100% | runs without final answer 925
aura          271 runs | 4.0 steps, 3.3 tool calls per run | tools: 5 | untrusted outputs 92% | runs without final answer 10
metaqa      1,800 runs | 3.3 steps, 2.4 tool calls per run | tools: 1 | untrusted outputs 0% | runs without final answer 0


## Two records of the same events: a PROV-O graph and a text log

**Graph** (rdflib): every run is a `prov:Activity` associated with its agent configuration (`prov:Agent`); every
agent step and every tool call is a `prov:Activity`; the user request, tool outputs and final answer are
`prov:Entity`s. A step `prov:used` the entities that arrived since the previous step and `prov:wasInformedBy` the
previous step; a tool call `prov:wasInformedBy` the step that issued it and `prov:wasAssociatedWith` the tool
(`prov:SoftwareAgent`); an output `prov:wasGeneratedBy` its call, the answer by the last step. Arguments are also
stored as key/value nodes so they can be queried.

**Text log:** one line per event with the same fields. The check below recovers the event list unchanged from the
text log (parser) and from the graph (traversal), so neither record has information the other lacks.

In [3]:
PROV = Namespace("http://www.w3.org/ns/prov#")
AU = Namespace("http://smash.example/audit#")
ID = Namespace("http://smash.example/id/")
NS = {"prov": PROV, "audit": AU, "id": ID, "rdf": RDF, "xsd": XSD}

# ---- join order for rdflib's SPARQL engine -------------------------------------------------------------------
# rdflib evaluates a basic graph pattern with nested loops after sorting its triple patterns by the number of
# unbound terms only, so `?call audit:tool "x" . ?arg audit:key "y" . ?call audit:arg ?arg` joins the first two
# patterns as a cross product; ordinary multi-join queries over a 50-run graph then run for minutes. This greedy
# planner, registered through rdflib's CUSTOM_EVALS hook, changes only the order (never the result): next comes the
# pattern with the most bound positions, preferring one that shares an already-bound variable, rdf:type last.
import rdflib.plugins.sparql as _sparql
from rdflib.plugins.sparql.evaluate import evalBGP as _evalBGP
from rdflib.term import Variable as _Var

def _greedy_order(patterns, ctx):
    isvar = lambda x: isinstance(x, (_Var, BNode))
    known = {x for t in patterns for x in t if isvar(x) and ctx[x] is not None}
    rest, out = list(patterns), []
    while rest:
        def key(t):
            bound = sum(1 for x in t if not isvar(x) or x in known)
            linked = any(isvar(x) and x in known for x in t)
            return (-bound, not linked, t[1] == RDF.type)
        best = min(rest, key=key)
        rest.remove(best); out.append(best)
        known |= {x for x in best if isvar(x)}
    return out

def _bgp_in_greedy_order(ctx, part):
    if part.name != "BGP":
        raise NotImplementedError
    return _evalBGP(ctx, _greedy_order(part.triples, ctx))

_sparql.CUSTOM_EVALS["nb24_greedy_bgp"] = _bgp_in_greedy_order


def scalar(v):
    return v if isinstance(v, str) else json.dumps(v)

def arg_pairs(args):
    # (key, value) for every scalar argument, every element of a list argument and every field of a dict argument
    try:
        d = json.loads(args)
    except Exception:
        return []
    out = []
    for k, v in (d.items() if isinstance(d, dict) else []):
        if isinstance(v, list):
            out += [(k, scalar(x)) for x in v if not isinstance(x, (list, dict))]
        elif isinstance(v, dict):
            out += [(f"{k}.{k2}", scalar(x)) for k2, x in v.items() if not isinstance(x, (list, dict))]
        else:
            out.append((k, scalar(v)))
    return out

def build_graph(runs):
    G = Graph()
    for p, n in NS.items():
        G.bind(p, n)
    add = G.add
    def node(i, *types):
        u = ID[i]
        add((u, AU.id, Literal(i)))
        for t in types:
            add((u, RDF.type, t))
        return u
    for r in runs:
        run, ag = node(r["id"], PROV.Activity, AU.Run), ID["agent-" + r["agent"]]
        add((run, AU.system, Literal(r["system"]))); add((run, PROV.wasAssociatedWith, ag))
        add((ag, RDF.type, PROV.Agent)); add((ag, RDF.type, AU.AgentConfig)); add((ag, AU.name, Literal(r["agent"])))
        q = r["request"]
        e = node(q["id"], PROV.Entity, AU.UserRequest)
        add((e, AU.inRun, run)); add((e, AU.trust, Literal(q["trust"]))); add((e, AU.content, Literal(q["text"])))
        prev = None
        for st in r["steps"]:
            s = node(f"{r['id']}.s{st['k']}", PROV.Activity, AU.Step)
            add((s, AU.inRun, run)); add((s, AU["index"], Literal(st["k"], datatype=XSD.integer)))
            add((s, PROV.wasAssociatedWith, ag))
            if prev is not None:
                add((s, PROV.wasInformedBy, prev))
            for u in st["used"]:
                add((s, PROV.used, ID[u]))
            for c in st["calls"]:
                cn, tool = node(c["id"], PROV.Activity, AU.ToolCall), ID["tool-" + c["tool"]]
                add((cn, AU.inRun, run)); add((cn, PROV.wasInformedBy, s)); add((cn, PROV.wasAssociatedWith, tool))
                add((tool, RDF.type, PROV.SoftwareAgent)); add((tool, RDF.type, AU.Tool)); add((tool, AU.name, Literal(c["tool"])))
                add((cn, AU.tool, Literal(c["tool"]))); add((cn, AU.args, Literal(c["args"])))
                for i, (k, v) in enumerate(arg_pairs(c["args"])):
                    b = ID[f"{c['id']}.arg{i}"]
                    add((cn, AU.arg, b)); add((b, AU.key, Literal(k))); add((b, AU.value, Literal(v)))
                if c["out"]:
                    o = node(c["out"]["id"], PROV.Entity, AU.ToolOutput)
                    add((o, AU.inRun, run)); add((o, PROV.wasGeneratedBy, cn))
                    add((o, AU.trust, Literal(c["out"]["trust"]))); add((o, AU.content, Literal(c["out"]["text"])))
            prev = s
        if r["answer"]:
            a = node(r["answer"]["id"], PROV.Entity, AU.FinalAnswer)
            add((a, AU.inRun, run)); add((a, PROV.wasGeneratedBy, ID[f"{r['id']}.s{r['answer']['step']}"]))
            add((a, AU.content, Literal(r["answer"]["text"])))
    return G

def render_log(runs):
    L = []
    for r in runs:
        L.append(f"RUN {r['id']} | system={r['system']} | agent={r['agent']}")
        q = r["request"]
        L.append(f"ENTITY {q['id']} | type=UserRequest | trust={q['trust']} | content={json.dumps(q['text'], ensure_ascii=False)}")
        for st in r["steps"]:
            inf = f"{r['id']}.s{st['k'] - 1}" if st["k"] > 1 else "-"
            L.append(f"STEP {r['id']}.s{st['k']} | index={st['k']} | informed_by={inf} | used=[{', '.join(st['used'])}]")
            for c in st["calls"]:
                L.append(f"TOOL_CALL {c['id']} | issued_by={r['id']}.s{st['k']} | tool={c['tool']} | args={c['args']}")
                if c["out"]:
                    o = c["out"]
                    L.append(f"ENTITY {o['id']} | type=ToolOutput | generated_by={c['id']} | trust={o['trust']} | "
                             f"content={json.dumps(o['text'], ensure_ascii=False)}")
        if r["answer"]:
            a = r["answer"]
            L.append(f"ENTITY {a['id']} | type=FinalAnswer | generated_by={r['id']}.s{a['step']} | "
                     f"content={json.dumps(a['text'], ensure_ascii=False)}")
        L.append(f"END RUN {r['id']}")
    return "\n".join(L)

# ---- the same-information check: recover the event lists from both records -------------------------------------
def runs_from_log(text):
    runs, r, step, calls = [], None, None, {}
    for line in text.split("\n"):
        kind, rest = line.split(" ", 1)
        if kind == "RUN":
            f = rest.split(" | ")
            r = {"id": f[0], "system": f[1][7:], "agent": f[2][6:], "steps": [], "answer": None}
        elif kind == "STEP":
            f = rest.split(" | ")
            u = f[3][len("used=["):-1]
            step = {"k": int(f[1][6:]), "used": u.split(", ") if u else [], "calls": []}
            r["steps"].append(step)
        elif kind == "TOOL_CALL":
            f = rest.split(" | ", 3)
            c = {"id": f[0], "tool": f[2][5:], "args": f[3][5:], "out": None}
            step["calls"].append(c); calls[c["id"]] = c
        elif kind == "ENTITY":
            f = rest.split(" | ")
            i, typ = f[0], f[1][5:]
            content = json.loads(rest.split(" | content=", 1)[1])
            if typ == "UserRequest":
                r["request"] = {"id": i, "text": content, "trust": f[2][6:]}
            elif typ == "ToolOutput":
                calls[f[2][13:]]["out"] = {"id": i, "text": content, "trust": f[3][6:]}
            else:
                r["answer"] = {"id": i, "step": int(f[2].rsplit(".s", 1)[1]), "text": content}
        elif kind == "END":
            runs.append(r)
    return runs

def runs_from_graph(G):
    v = lambda s, p: G.value(s, p)
    lid = lambda u: str(v(u, AU.id))
    out = []
    for run in sorted(G.subjects(RDF.type, AU.Run), key=str):
        parts = defaultdict(list)
        for x in G.subjects(AU.inRun, run):
            parts[next(t for t in G.objects(x, RDF.type) if t != PROV.Entity and t != PROV.Activity)].append(x)
        req = parts[AU.UserRequest][0]
        r = {"id": lid(run), "system": str(v(run, AU.system)), "agent": str(v(v(run, PROV.wasAssociatedWith), AU.name)),
             "steps": [], "answer": None,
             "request": {"id": lid(req), "text": str(v(req, AU.content)), "trust": str(v(req, AU.trust))}}
        for s in sorted(parts[AU.Step], key=lambda s: int(v(s, AU["index"]))):
            calls = [c for c in G.subjects(PROV.wasInformedBy, s) if (c, RDF.type, AU.ToolCall) in G]
            calls.sort(key=lambda c: int(lid(c).rsplit(".", 1)[1]))   # (the next step is informed by this step too)
            st = {"k": int(v(s, AU["index"])), "used": sorted(lid(u) for u in G.objects(s, PROV.used)), "calls": []}
            for c in calls:
                o = next(G.subjects(PROV.wasGeneratedBy, c), None)
                st["calls"].append({"id": lid(c), "tool": str(v(c, AU.tool)), "args": str(v(c, AU.args)),
                                    "out": None if o is None else {"id": lid(o), "text": str(v(o, AU.content)),
                                                                   "trust": str(v(o, AU.trust))}})
            r["steps"].append(st)
        for a in parts[AU.FinalAnswer]:
            r["answer"] = {"id": lid(a), "step": int(v(v(a, PROV.wasGeneratedBy), AU["index"])), "text": str(v(a, AU.content))}
        out.append(r)
    return out

def canon(runs):
    return json.dumps(sorted(runs, key=lambda r: r["id"]), sort_keys=True, ensure_ascii=False)

## Audit logs of 1, 10, 50 and 200 runs, and the questions

For each system and log length, disjoint sets of runs form the logs. Each log gets its share of the 111 questions of
its (system, length) cell, drawn round-robin over the templates that apply; the parameters (tool, argument, run, call,
value) are drawn from the log itself, and about one in five count/list questions asks about a tool or value from the
same system that does **not** occur in the log (gold 0 or empty). The question order is fixed: question 1 of every
log, then question 2 of every log, …, so the first 25 per length (test mode) cover all systems.

In [4]:
rng = random.Random(24)
TEMPLATES = {"metaqa": ["count_runs", "list_runs", "count_calls", "cause", "source"],
             "aura": ["count_runs", "list_runs", "count_calls", "cause", "source", "first_untrusted", "after_untrusted"],
             "injecagent": ["count_runs", "list_runs", "count_calls", "cause", "source", "first_untrusted", "after_untrusted"]}
KIND = {"count_runs": "count", "count_calls": "count", "first_untrusted": "step",
        "list_runs": "list", "cause": "list", "source": "list", "after_untrusted": "list"}

def all_calls(r):
    return [(st, c) for st in r["steps"] for c in st["calls"]]

def first_untrusted(r):
    bad = {c["out"]["id"] for _, c in all_calls(r) if c["out"] and c["out"]["trust"] == "untrusted"}
    return next((st["k"] for st in r["steps"] if set(st["used"]) & bad), None)

def calls_with(r, tool, key=None, val=None):
    return [c for _, c in all_calls(r) if c["tool"] == tool and (key is None or (key, val) in arg_pairs(c["args"]))]

VOCAB = {}                                         # per system: tools and short (tool, key, value) triples seen anywhere
for s, rs in RUNS.items():
    trip = Counter((c["tool"], k, v) for r in rs for _, c in all_calls(r) for k, v in arg_pairs(c["args"])
                   if 0 < len(v) <= 60 and "." not in k)
    VOCAB[s] = {"tools": sorted({c["tool"] for r in rs for _, c in all_calls(r)}),
                "triples": sorted(t for t, n in trip.items() if n >= 2)}

TOKEN = re.compile(r"[\w@.\-]{5,}")

def answer_values(r):
    a = r["answer"]["text"] if r["answer"] else ""
    if r["system"] == "metaqa":
        try:
            vals = [str(x) for x in json.loads(a).get("answers", [])]
        except Exception:
            vals = []
    elif r["system"] == "aura":
        vals = re.findall(r"\b(?:tkt|cmt|doc)_\d+", a)
    else:
        vals = [t.strip(".-") for t in TOKEN.findall(a) if re.search(r"[\d@A-Z]", t)]
    outs = [c["out"]["text"].lower() for _, c in all_calls(r) if c["out"]]
    return sorted({v for v in vals if len(v) >= 3 and any(v.lower() in o for o in outs)})

def make_question(t, log, system):
    runs = log["runs"]
    pick_absent = rng.random() < 0.2
    if t in ("count_runs", "list_runs"):
        with_arg = t == "list_runs" or system == "metaqa" or rng.random() < 0.5
        if with_arg:
            present = sorted({(c["tool"], k, v) for r in runs for _, c in all_calls(r) for k, v in arg_pairs(c["args"])
                              if 0 < len(v) <= 60 and "." not in k})
            pool = [x for x in VOCAB[system]["triples"] if x not in set(present)] if pick_absent else present
            if not pool:
                return None
            tool, key, val = rng.choice(pool)
        else:
            present = sorted({c["tool"] for r in runs for _, c in all_calls(r)})
            # prefer tools called after step 1 in InjecAgent (the attacker tools), otherwise the user tool dominates
            later = sorted({c["tool"] for r in runs for st, c in all_calls(r) if st["k"] > 1})
            pool = [x for x in VOCAB[system]["tools"] if x not in present] if pick_absent else (later if later and rng.random() < 0.6 else present)
            if not pool:
                return None
            tool, key, val = rng.choice(pool), None, None
        hit = sorted(r["id"] for r in runs if calls_with(r, tool, key, val))
        cond = f"the tool `{tool}`" + (f" with the argument `{key}` set to (or containing) `{val}`" if key else "")
        if t == "count_runs":
            return {"q": f"How many runs in this log called {cond} at least once?", "gold": str(len(hit))}
        return {"q": f"Which runs in this log called {cond}? List the run ids.", "gold": hit}
    if t == "after_untrusted":
        hit = sorted(r["id"] for r in runs if first_untrusted(r) is not None
                     and any(st["calls"] for st in r["steps"] if st["k"] >= first_untrusted(r)))
        return {"q": "Which runs issued at least one tool call from a step at or after the first step of that run that "
                     "used an untrusted entity? List the run ids.", "gold": hit}
    r = rng.choice(runs)
    if t == "count_calls":
        return {"q": f"How many tool calls did run `{r['id']}` make in total (including calls whose output was not "
                     f"recorded)?", "gold": str(len(all_calls(r)))}
    if t == "first_untrusted":
        k = first_untrusted(r)
        return {"q": f"In run `{r['id']}`, what is the index of the first agent step that used an untrusted entity? "
                     f"Answer the step number, or none if no step used one.", "gold": "none" if k is None else str(k)}
    if t == "cause":
        sig = Counter((c["tool"], c["args"]) for _, c in all_calls(r))
        cands = [(st, c) for st, c in all_calls(r) if sig[(c["tool"], c["args"])] == 1 and len(c["args"]) <= 300]
        later = [x for x in cands if x[0]["k"] > 1]
        if not cands:
            return None
        st, c = rng.choice(later if later and rng.random() < 0.7 else cands)
        return {"q": f"In run `{r['id']}`, the agent called `{c['tool']}` with the arguments {c['args']}. Which entities "
                     f"did the agent step that issued this call use? List their ids.", "gold": sorted(st["used"])}
    if t == "source":
        vals = answer_values(r)
        if not vals:
            return None
        v = rng.choice(vals)
        hit = sorted(c["out"]["id"] for _, c in all_calls(r) if c["out"] and v.lower() in c["out"]["text"].lower())
        return {"q": f"In run `{r['id']}`, the final answer mentions `{v}`. Which tool outputs of that run contain "
                     f"`{v}`? List their ids.", "gold": hit}

LOGS, QUESTIONS = {}, []
for L in LENGTHS:
    for system in RUNS:
        order = sorted(RUNS[system], key=lambda r: r["id"])
        rng.shuffle(order)
        per_log = math.ceil(Q_PER_CELL / N_LOGS[L])
        cell = []
        for li in range(N_LOGS[L]):
            lid = f"{system}-L{L}-{li:02d}"
            # Aura has 271 runs: its second 200-run log wraps around and shares 129 runs with the first
            log = {"id": lid, "system": system, "length": L, "runs": (order * 2)[li * L:(li + 1) * L]}
            LOGS[lid] = log
            temps = TEMPLATES[system][:]
            rng.shuffle(temps)
            qs, seen = [], set()
            for i in range(300):                     # round-robin over the templates; skip the ones that do not apply
                if len(qs) == per_log:
                    break
                t = temps[i % len(temps)]
                x = make_question(t, log, system)
                if x is None or x["q"] in seen:
                    continue
                seen.add(x["q"])
                qs.append({"log": lid, "system": system, "length": L, "template": t, "kind": KIND[t], "rank": len(qs), **x})
            cell += qs
        cell.sort(key=lambda q: (q["rank"], q["log"]))
        QUESTIONS += cell[:Q_PER_CELL]
order_key = {s: i for i, s in enumerate(RUNS)}
QUESTIONS.sort(key=lambda q: (q["length"], q["rank"], int(q["log"].rsplit("-", 1)[1]), order_key[q["system"]]))
for i, q in enumerate(QUESTIONS):
    q["qid"] = f"q{i:04d}"
os.makedirs(BASE, exist_ok=True)
json.dump(QUESTIONS, open(f"{BASE}/questions.json", "w"), indent=0, ensure_ascii=False)
print(f"{len(QUESTIONS)} questions, {len(LOGS)} logs")
print(Counter((q["length"], q["system"]) for q in QUESTIONS))
print(Counter(q["template"] for q in QUESTIONS))
print("list questions with an empty gold set:", sum(q["kind"] == "list" and not q["gold"] for q in QUESTIONS),
      "| count questions with gold 0:", sum(q["kind"] == "count" and q["gold"] == "0" for q in QUESTIONS))
if TEST:
    QUESTIONS = [q for L in LENGTHS for q in [x for x in QUESTIONS if x["length"] == L][:N_PER_LEN[L]]]
    print(f"test mode: {len(QUESTIONS)} questions ({Counter(q['system'] for q in QUESTIONS)})")
for q in QUESTIONS[:4]:
    print(f"  [{q['log']}] {q['q']}  ->  {q['gold']}")

1332 questions, 165 logs
Counter({(1, 'injecagent'): 111, (1, 'aura'): 111, (1, 'metaqa'): 111, (10, 'injecagent'): 111, (10, 'aura'): 111, (10, 'metaqa'): 111, (50, 'injecagent'): 111, (50, 'aura'): 111, (50, 'metaqa'): 111, (200, 'injecagent'): 111, (200, 'aura'): 111, (200, 'metaqa'): 111})
Counter({'cause': 245, 'list_runs': 232, 'count_runs': 229, 'count_calls': 224, 'source': 180, 'first_untrusted': 153, 'after_untrusted': 69})
list questions with an empty gold set: 68 | count questions with gold 0: 41
  [injecagent-L1-00] In run `inj-none-ds-enhanced-53`, what is the index of the first agent step that used an untrusted entity? Answer the step number, or none if no step used one.  ->  2
  [aura-L1-00] Which runs in this log called the tool `semantic_search` with the argument `label` set to (or containing) `Ticket`? List the run ids.  ->  ['aura-q144']
  [metaqa-L1-00] How many tool calls did run `mqa-extracted-2hop-4105` make in total (including calls whose output was not recorde

In [5]:
USED_LOGS = sorted({q["log"] for q in QUESTIONS})
TEXT, GRAPH = {}, {}
t0 = time.time()
for lid in USED_LOGS:
    runs = LOGS[lid]["runs"]
    TEXT[lid], GRAPH[lid] = render_log(runs), build_graph(runs)
    assert canon(runs_from_log(TEXT[lid])) == canon(runs), lid
    assert canon(runs_from_graph(GRAPH[lid])) == canon(runs), lid
print(f"{len(USED_LOGS)} logs built in {time.time() - t0:.0f} s; every event list was recovered unchanged from the "
      f"text log and from the PROV graph (same information).")
for L in LENGTHS:
    ls = [l for l in USED_LOGS if LOGS[l]["length"] == L]
    print(f"  {L:>2}-run logs: {np.mean([len(TEXT[l]) for l in ls]) / 4:,.0f} tokens of text (≈ chars/4), "
          f"{np.mean([len(TEXT[l].splitlines()) for l in ls]):,.0f} lines, {np.mean([len(GRAPH[l]) for l in ls]):,.0f} triples")
ex = LOGS[USED_LOGS[0]]["runs"][0]["id"]
print("\nExample (the start of one run in the text log):")
print("\n".join(l[:200] for l in TEXT[USED_LOGS[0]].splitlines()[:8]))

165 logs built in 2 s; every event list was recovered unchanged from the text log and from the PROV graph (same information).
   1-run logs: 577 tokens of text (≈ chars/4), 12 lines, 116 triples
  10-run logs: 5,380 tokens of text (≈ chars/4), 115 lines, 937 triples
  50-run logs: 28,180 tokens of text (≈ chars/4), 589 lines, 5,128 triples
  200-run logs: 109,565 tokens of text (≈ chars/4), 2,314 lines, 19,404 triples

Example (the start of one run in the text log):
RUN aura-q144 | system=aura | agent=aura-template
ENTITY aura-q144.req | type=UserRequest | trust=trusted | content="What is the status of the ticket 'Aura fails to answer even if correct source is shown', and who is it assigned to?"
STEP aura-q144.s1 | index=1 | informed_by=- | used=[aura-q144.req]
TOOL_CALL aura-q144.c1.1 | issued_by=aura-q144.s1 | tool=semantic_search | args={"query":"Aura fails to answer even if correct source is shown","label":"Ticket","k":5}
ENTITY aura-q144.o1.1 | type=ToolOutput | generated_by=aura-

## Shared instructions (identical in all three conditions)

In [6]:
RECORD_DOC = """The record covers a set of runs of an LLM agent. Each run has a user request, a sequence of agent steps (LLM calls) numbered 1, 2, ..., the tool calls each step issued with their arguments, the output of each call (if one was recorded), and the final answer if the run ended with one. Semantics:
- A step USED the entities that arrived since the previous step: the user request for step 1, otherwise the outputs of the previous step's tool calls. Each step after the first was INFORMED BY the previous step.
- A tool call was ISSUED BY (informed by) the step that made it; its output was GENERATED BY the call. The final answer was generated by the last step.
- The user request and every tool output carry a trust label: trusted or untrusted.
- Ids: <run>.req (user request), <run>.sK (step K), <run>.cK.J (J-th tool call issued by step K), <run>.oK.J (output of that call), <run>.ans (final answer). Run ids look like aura-q000, mqa-kb-1hop-5820, inj-none-dh-base-0.
- Contents (requests, outputs, answers) are truncated to 400 characters."""

ANSWER_RULES = """Return JSON {"answer": [...]}, a list of strings:
- a count: one number, e.g. ["3"] (["0"] if none);
- a step number: e.g. ["2"], or ["none"];
- a list of runs or entities: every matching id, exactly as written in the record, e.g. ["aura-q001", "aura-q007"], or [] if there are none."""

LOG_FORMAT = """The text log has one line per event:
RUN <run> | system=... | agent=<agent configuration>
ENTITY <run>.req | type=UserRequest | trust=... | content="..."
STEP <run>.sK | index=K | informed_by=<previous step or -> | used=[<entity ids>]
TOOL_CALL <run>.cK.J | issued_by=<run>.sK | tool=<tool name> | args=<JSON arguments>
ENTITY <run>.oK.J | type=ToolOutput | generated_by=<run>.cK.J | trust=... | content="..."
ENTITY <run>.ans | type=FinalAnswer | generated_by=<run>.sK | content="..."
END RUN <run>"""

class Answer(BaseModel):
    answer: list[str]

class Query(BaseModel):
    sparql: str = Field(description="one SPARQL 1.1 SELECT query")

def norm_id(s):
    s = str(s).strip().strip("`'\"<> ").strip()
    s = re.sub(r"^(id:|https?://\S+/)", "", s)
    return s.lower()

def to_int(xs):
    # the last integer of the first item that has one ("3", "3 runs", "step 2", "aura-q000.s2"); none/no/zero -> 0
    for x in xs:
        if re.fullmatch(r"\s*(none|no|zero|no runs)\s*", str(x), re.I):
            return 0
        m = re.findall(r"\d+", str(x))
        if m:
            return int(m[-1])
    return None

def score(q, parsed):
    pred = (parsed or {}).get("answer") if isinstance(parsed, dict) else None
    if pred is None:
        return {"acc": 0.0, "rec": 0.0 if q["kind"] == "list" and q["gold"] else float("nan"), "prec": float("nan"), "ok": 0}
    if q["kind"] == "count":
        p = 0 if not pred else to_int(pred)
        return {"acc": float(p == int(q["gold"])), "rec": float("nan"),
                "prec": float("nan"), "ok": 1}
    if q["kind"] == "step":
        none = not pred or any(str(x).strip().lower() in ("none", "null", "") for x in pred)
        p = None if none else to_int(pred)
        g = None if q["gold"] == "none" else int(q["gold"])
        return {"acc": float(p == g), "rec": float("nan"), "prec": float("nan"), "ok": 1}
    P = {norm_id(x) for x in pred if norm_id(x) not in ("", "none")}
    G = {norm_id(x) for x in q["gold"]}
    return {"acc": float(P == G), "rec": len(P & G) / len(G) if G else float("nan"),
            "prec": len(P & G) / len(P) if P else float("nan"), "ok": 1}

## Condition (a): an LLM reads the whole text log (one Batch job)

In [7]:
FULL_SYS = ("You audit the actions of LLM agents from their audit log. Answer the question from the log only.\n\n"
            + RECORD_DOC + "\n\n" + LOG_FORMAT + "\n\n" + ANSWER_RULES)
def full_body(q, question=None, max_tokens=1500):
    return LLM.body([{"role": "system", "content": FULL_SYS},
                     {"role": "user", "content": f"AUDIT LOG ({LOGS[q['log']]['length']} runs)\n{TEXT[q['log']]}\n\n"
                                                 f"Question: {question or q['q']}"}], max_tokens=max_tokens, schema=Answer)

# Context check: every log of 50 or more runs is sent once, synchronously, with a trivial question and a 16-token
# limit; the API either reports the prompt size or rejects it as too long. Nothing is truncated: a log that does not
# fit is recorded, and condition (a) is not run on its questions (reported as "not runnable"). The probe results are
# kept in a JSON file (an error row in the LLM cache would be re-sent on every replay).
PROBE_PATH = f"{BASE}/context_probe.json"
PROBE = json.load(open(PROBE_PATH)) if os.path.exists(PROBE_PATH) else {}
FITS = {}
for lid in USED_LOGS:
    if LOGS[lid]["length"] < 50:
        FITS[lid] = True; continue
    body = full_body({"log": lid}, "Reply with an empty answer list.", max_tokens=16)
    h = LLM._hash(body)
    if PROBE.get(lid, {}).get("h") != h:
        if MODE == "replay":
            raise RuntimeError(f"context probe for {lid} not cached - run with SMASH_MODE=test or batch")
        try:
            u = llm.client.chat.completions.create(**body).usage
            PROBE[lid] = {"h": h, "fits": True, "prompt_tokens": u.prompt_tokens, "out": u.completion_tokens}
        except Exception as e:
            msg = str(e)
            if not re.search(r"context|too long|maximum|token", msg, re.I):
                raise
            PROBE[lid] = {"h": h, "fits": False, "prompt_tokens": None, "out": 0, "error": msg[:300]}
        json.dump(PROBE, open(PROBE_PATH, "w"), indent=1)
    FITS[lid] = PROBE[lid]["fits"]
for L in LENGTHS:
    ps = [PROBE[l] for l in USED_LOGS if LOGS[l]["length"] == L and l in PROBE]
    if ps:
        tok = [p["prompt_tokens"] for p in ps if p["fits"]]
        print(f"{L}-run logs: {sum(p['fits'] for p in ps)} of {len(ps)} fit the context"
              + (f"; prompt {min(tok):,}-{max(tok):,} tokens" if tok else "")
              + "".join(f"\n  does not fit: {l} ({PROBE[l]['error'][:150]})" for l in USED_LOGS if l in PROBE and not PROBE[l]["fits"]))
PROBE_USD = sum((p["prompt_tokens"] or 0) * 0.10 / 1e6 + p["out"] * 0.50 / 1e6 for l, p in PROBE.items() if l in USED_LOGS)

runnable = [q for q in QUESTIONS if FITS[q["log"]]]
res = llm.run("nb24_fulllog", {q["qid"]: full_body(q) for q in runnable}, parse=Answer)
RES = {"full": {q["qid"]: ({"parsed": res[q["qid"]]["parsed"], "tokens": res[q["qid"]].get("in", 0) + res[q["qid"]].get("out", 0)}
                           if q["qid"] in res else None) for q in QUESTIONS}}
print("full-log answers:", len(res), "| not runnable (log exceeds the context):", len(QUESTIONS) - len(runnable),
      "| unparsed:", sum(r["parsed"] is None for r in res.values()))

50-run logs: 12 of 12 fit the context; prompt 21,762-48,762 tokens
200-run logs: 6 of 6 fit the context; prompt 86,645-181,632 tokens


full-log answers: 1332 | not runnable (log exceeds the context): 0 | unparsed: 1


## Condition (b): an agent searches the log with `grep` and `read_lines`

Up to three tool-using steps, then a fourth step without tools that must answer. Every step is one `llm.run` over all
questions still open (one Batch job per step), as in Notebooks 1.1b and 1.2b.

In [8]:
MAX_TOOL_STEPS, GREP_MAX, READ_MAX = 3, 40, 40
SEARCH_TOOLS = [
    {"type": "function", "function": {"name": "grep", "description": (
        "Search the audit log with a Python regular expression (case-insensitive). Returns the number of matching "
        "lines and up to max_matches of them, each prefixed with its line number."),
        "parameters": {"type": "object", "properties": {"pattern": {"type": "string"},
                                                        "max_matches": {"type": "integer"}}, "required": ["pattern"]}}},
    {"type": "function", "function": {"name": "read_lines", "description": "Read up to 40 consecutive lines of the log.",
        "parameters": {"type": "object", "properties": {"start": {"type": "integer"}, "count": {"type": "integer"}},
                       "required": ["start"]}}}]

def search_tool(lines, name, args):
    try:
        a = json.loads(args)
        if name == "grep":
            try:
                rx = re.compile(a["pattern"], re.I)
            except re.error:
                rx = re.compile(re.escape(a["pattern"]), re.I)
            hits = [i for i, l in enumerate(lines, 1) if rx.search(l)]
            m = max(1, min(int(a.get("max_matches") or GREP_MAX), GREP_MAX))
            return f"{len(hits)} matching lines" + (f" (first {m} shown)" if len(hits) > m else "") + ":\n" + \
                   "\n".join(f"{i}: {lines[i - 1]}" for i in hits[:m])
        if name == "read_lines":
            s = max(1, int(a.get("start", 1))); n = max(1, min(int(a.get("count") or READ_MAX), READ_MAX))
            return "\n".join(f"{i}: {lines[i - 1]}" for i in range(s, min(len(lines), s + n - 1) + 1)) or "(no such lines)"
        return f"unknown tool {name}"
    except Exception as e:
        return f"error: {type(e).__name__}: {e}"

SEARCH_SYS = ("You audit the actions of LLM agents from their audit log. The log is not shown to you; search it with the "
              "tools grep and read_lines. You have at most " + str(MAX_TOOL_STEPS) + " rounds of tool calls (several "
              "calls per round are allowed), then you must answer from what you found.\n\n"
              + RECORD_DOC + "\n\n" + LOG_FORMAT + "\n\n" + ANSWER_RULES)
LINES = {lid: TEXT[lid].split("\n") for lid in USED_LOGS}
state = {q["qid"]: {"msgs": [{"role": "system", "content": SEARCH_SYS},
                             {"role": "user", "content": f"The audit log has {LOGS[q['log']]['length']} runs and "
                                                         f"{len(LINES[q['log']])} lines.\n\nQuestion: {q['q']}"}],
                    "parsed": None, "done": False, "tokens": 0, "calls": 0} for q in QUESTIONS}
QLOG = {q["qid"]: q["log"] for q in QUESTIONS}
for step in range(MAX_TOOL_STEPS + 1):
    active = [k for k, s in state.items() if not s["done"]]
    if not active:
        break
    last = step == MAX_TOOL_STEPS
    reqs = {}
    for k in active:
        msgs = state[k]["msgs"] + ([{"role": "user", "content": "No more tool calls. Give your final answer now."}] if last else [])
        reqs[k] = LLM.body(msgs, max_tokens=1500, schema=Answer, tools=None if last else SEARCH_TOOLS)
    res = llm.run(f"nb24_search_step{step + 1}", reqs, parse=Answer)
    for k in active:
        r, s = res[k], state[k]
        s["tokens"] += r.get("in", 0) + r.get("out", 0)
        if not r.get("tool_calls") or last:
            s["parsed"], s["done"] = r["parsed"], True
            continue
        s["msgs"].append({"role": "assistant", "content": r["text"] or None, "tool_calls": [
            {"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["arguments"]}} for t in r["tool_calls"]]})
        for t in r["tool_calls"]:
            s["calls"] += 1
            s["msgs"].append({"role": "tool", "tool_call_id": t["id"], "content": search_tool(LINES[QLOG[k]], t["name"], t["arguments"])})
    print(f"search step {step + 1}: {len(active)} agents active")
RES["search"] = {k: {"parsed": s["parsed"], "tokens": s["tokens"]} for k, s in state.items()}
print(f"tool calls per question: {np.mean([s['calls'] for s in state.values()]):.1f}")

search step 1: 1332 agents active


search step 2: 1332 agents active
search step 3: 357 agents active
search step 4: 122 agents active
tool calls per question: 1.4


## Condition (c): the LLM writes a SPARQL query over the PROV graph and answers from its result

The query is run with rdflib on the log's graph (prefixes `prov:`, `audit:`, `id:` are pre-bound; join order by the
greedy planner defined with the graph; 60 s limit per query). A query that raises an error or times out gets one repair call with the error message. The answering call sees the question, the query and its result
(at most 100 rows), not the graph.

In [9]:
SCHEMA = """PROV-O graph of the agent runs (prefixes are pre-bound: prov: <http://www.w3.org/ns/prov#>, audit: <http://smash.example/audit#>, id: <http://smash.example/id/>, rdf:, xsd:). Every node except argument nodes has audit:id "<its id string>"; node IRIs are id:<id>, e.g. id:aura-q000.s2.
Classes and properties:
- audit:Run (a prov:Activity): audit:system "aura|metaqa|injecagent"; prov:wasAssociatedWith ?agentConfig
- audit:AgentConfig (a prov:Agent): audit:name "aura-template" ...
- audit:UserRequest (a prov:Entity): audit:inRun ?run; audit:trust "trusted"|"untrusted"; audit:content "..."
- audit:Step (a prov:Activity): audit:inRun ?run; audit:index K (xsd:integer); prov:wasInformedBy ?previousStep; prov:used ?entity (request or tool outputs); prov:wasAssociatedWith ?agentConfig
- audit:ToolCall (a prov:Activity): audit:inRun ?run; prov:wasInformedBy ?issuingStep; prov:wasAssociatedWith ?tool; audit:tool "<tool name>"; audit:args "<JSON arguments>"; audit:arg ?a . ?a audit:key "<argument name>"; audit:value "<value as string>" (one ?a per scalar argument, per list element, and per field 'name.field' of an object argument)
- audit:Tool (a prov:SoftwareAgent): audit:name "<tool name>"
- audit:ToolOutput (a prov:Entity): audit:inRun ?run; prov:wasGeneratedBy ?toolCall; audit:trust "trusted"|"untrusted"; audit:content "..."
- audit:FinalAnswer (a prov:Entity): audit:inRun ?run; prov:wasGeneratedBy ?lastStep; audit:content "..."
Tip: return audit:id values (strings) rather than IRIs; use COUNT(DISTINCT ...) for counts; use CONTAINS(LCASE(...)) for text."""

GRAPH_Q_SYS = ("You audit the actions of LLM agents. Their runs are recorded as a W3C PROV-O graph. Write ONE SPARQL 1.1 "
               "SELECT query whose result answers the question (it will be run on the graph, and the answer will be "
               "written from its result only).\n\n" + SCHEMA + "\n\n" + RECORD_DOC)
GRAPH_A_SYS = ("You audit the actions of LLM agents. A SPARQL query was run on the PROV-O graph of the agent runs; answer "
               "the question from the query result only.\n\n" + RECORD_DOC + "\n\n" + ANSWER_RULES)
MAX_ROWS = 100

import signal
class QueryTimeout(Exception):
    pass
def _alarm(*_):
    raise QueryTimeout("the query ran longer than 60 s")
signal.signal(signal.SIGALRM, _alarm)

def run_sparql(lid, text):
    signal.alarm(60)
    try:
        res = GRAPH[lid].query(text, initNs=NS)
        rows = list(res)
        signal.alarm(0)
        head = [str(v) for v in res.vars] if res.vars else []
        cell = lambda x: "" if x is None else (str(x).split("/id/")[-1] if isinstance(x, URIRef) else clip(str(x), 200))
        body = [" | ".join(cell(x) for x in row) for row in rows[:MAX_ROWS]]
        more = f"\n({len(rows)} rows; first {MAX_ROWS} shown)" if len(rows) > MAX_ROWS else f"\n({len(rows)} rows)"
        return None, " | ".join(head) + "\n" + "\n".join(body) + more
    except Exception as e:
        return f"{type(e).__name__}: {str(e)[:300]}", None
    finally:
        signal.alarm(0)

def q_msgs(q):
    return [{"role": "system", "content": GRAPH_Q_SYS}, {"role": "user", "content": f"Question: {q['q']}"}]

res = llm.run("nb24_graph_query", {q["qid"]: LLM.body(q_msgs(q), max_tokens=800, schema=Query) for q in QUESTIONS}, parse=Query)
G_STATE = {}
for q in QUESTIONS:
    r = res[q["qid"]]
    sq = (r["parsed"] or {}).get("sparql", "")
    err, out = run_sparql(q["log"], sq)
    G_STATE[q["qid"]] = {"sparql": sq, "err": err, "out": out, "tokens": r.get("in", 0) + r.get("out", 0), "repaired": False}
bad = [q for q in QUESTIONS if G_STATE[q["qid"]]["err"]]
print(f"queries: {len(QUESTIONS)} | raised an error: {len(bad)}")
if bad:
    reqs = {q["qid"]: LLM.body(q_msgs(q) + [{"role": "assistant", "content": json.dumps({"sparql": G_STATE[q["qid"]]["sparql"]})},
                                            {"role": "user", "content": f"The query failed: {G_STATE[q['qid']]['err']}\nWrite a corrected query."}],
                               max_tokens=800, schema=Query) for q in bad}
    res = llm.run("nb24_graph_repair", reqs, parse=Query)
    for q in bad:
        st, r = G_STATE[q["qid"]], res[q["qid"]]
        st["sparql"] = (r["parsed"] or {}).get("sparql", "")
        st["err"], st["out"] = run_sparql(q["log"], st["sparql"])
        st["tokens"] += r.get("in", 0) + r.get("out", 0); st["repaired"] = True
    print(f"after one repair: {sum(bool(G_STATE[q['qid']]['err']) for q in QUESTIONS)} queries still fail")
reqs = {}
for q in QUESTIONS:
    st = G_STATE[q["qid"]]
    result = st["out"] if st["err"] is None else f"(the query failed: {st['err']})"
    reqs[q["qid"]] = LLM.body([{"role": "system", "content": GRAPH_A_SYS},
                               {"role": "user", "content": f"Question: {q['q']}\n\nSPARQL query:\n{st['sparql']}\n\nResult:\n{result}"}],
                              max_tokens=1500, schema=Answer)
res = llm.run("nb24_graph_answer", reqs, parse=Answer)
RES["graph"] = {}
for q in QUESTIONS:
    G_STATE[q["qid"]]["tokens"] += res[q["qid"]].get("in", 0) + res[q["qid"]].get("out", 0)
    RES["graph"][q["qid"]] = {"parsed": res[q["qid"]]["parsed"], "tokens": G_STATE[q["qid"]]["tokens"]}
print("graph answers:", len(res))

queries: 1332 | raised an error: 5
after one repair: 2 queries still fail
graph answers: 1332


## Results with 95% confidence intervals

Accuracy per condition (Wilson interval) and paired bootstrap intervals of the differences graph − full log and
graph − search agent, over all questions and per log length. Completeness and precision are averaged over list
questions with a non-empty gold set.

In [10]:
CONDS = ("full", "search", "graph")
NAMES = {"full": "full log", "search": "search agent", "graph": "PROV graph"}
NA = {"acc": float("nan"), "rec": float("nan"), "prec": float("nan"), "ok": 0}    # condition (a) not runnable
SC = {c: {q["qid"]: score(q, RES[c][q["qid"]]["parsed"]) if RES[c][q["qid"]] else NA for q in QUESTIONS} for c in CONDS}
tok = lambda c, qs: np.mean([RES[c][q["qid"]]["tokens"] for q in qs if RES[c][q["qid"]]]) if any(RES[c][q["qid"]] for q in qs) else float("nan")
SUMMARY = {}

def table(label, qs):
    if not qs:
        return
    n = len(qs)
    acc = {c: [SC[c][q["qid"]]["acc"] for q in qs] for c in CONDS}
    ran = [i for i, x in enumerate(acc["full"]) if not np.isnan(x)]          # questions condition (a) could run on
    d_full = paired_boot([acc["full"][i] for i in ran], [acc["graph"][i] for i in ran]) if ran else (float("nan"),) * 3
    d_search = paired_boot(acc["search"], acc["graph"])
    SUMMARY[label] = {"n": n, "n_full": len(ran), "acc": {c: np.nanmean(acc[c]) if c != "full" or ran else float("nan") for c in CONDS},
                      "g-full": d_full, "g-search": d_search, "tokens": {c: tok(c, qs) for c in CONDS}}
    ci = lambda t: "not runnable" if np.isnan(t[0]) else f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]"
    note = f"  (full log on {len(ran)} of {n})" if len(ran) < n else ""
    print(f"{label:<26}{n:>5}" + "".join(f"{SUMMARY[label]['acc'][c]:>9.1%}" for c in CONDS) + f"{ci(d_full):>28}{ci(d_search):>28}{note}")

print(f"{'accuracy':<26}{'n':>5}" + "".join(f"{c:>9}" for c in CONDS)
      + f"{'graph - full [95% CI]':>28}{'graph - search [95% CI]':>28}")
table("all questions", QUESTIONS)
for L in LENGTHS:
    table(f"{L}-run logs", [q for q in QUESTIONS if q["length"] == L])
for s in RUNS:
    table(f"system {s}", [q for q in QUESTIONS if q["system"] == s])
for t in TEMPLATES["aura"]:
    table(f"template {t}", [q for q in QUESTIONS if q["template"] == t])
for c in CONDS:
    xs = [SC[c][q["qid"]]["acc"] for q in QUESTIONS if not np.isnan(SC[c][q["qid"]]["acc"])]
    lo, hi = wilson(sum(xs), len(xs))
    print(f"  {NAMES[c]:<13} accuracy {np.mean(xs):.1%} [Wilson {lo:.1%}, {hi:.1%}] on {len(xs)} questions")

accuracy                      n     full   search    graph       graph - full [95% CI]     graph - search [95% CI]
all questions              1332    87.5%    96.7%    93.5%        +5.9% [+3.7%, +8.1%]        -3.2% [-4.9%, -1.7%]


1-run logs                  333    93.7%    99.4%    94.6%        +0.9% [-2.7%, +4.5%]        -4.8% [-7.5%, -2.4%]
10-run logs                 333    89.2%    96.1%    94.6%        +5.4% [+1.2%, +9.6%]        -1.5% [-4.8%, +1.8%]
50-run logs                 333    86.5%    96.4%    91.0%        +4.5% [-0.6%, +9.3%]        -5.4% [-9.0%, -1.8%]
200-run logs                333    80.8%    94.9%    93.7%      +12.9% [+7.8%, +18.0%]        -1.2% [-4.5%, +2.1%]
system injecagent           444    89.6%    97.5%    96.2%        +6.5% [+3.2%, +9.9%]        -1.4% [-3.6%, +0.9%]
system aura                 444    77.9%    92.8%    88.5%      +10.6% [+5.4%, +15.5%]        -4.3% [-8.1%, -0.5%]
system metaqa               444    95.0%    99.8%    95.7%        +0.7% [-2.0%, +3.4%]        -4.1% [-6.1%, -2.3%]
template count_runs         229    80.3%    93.9%    98.7%     +18.3% [+13.1%, +23.6%]        +4.8% [+1.3%, +8.3%]
template list_runs          232    87.5%    99.6%    99.6%      +12.1% [+7.8%, +

template cause              245    97.1%    98.8%    73.5%     -23.7% [-29.8%, -18.4%]     -25.3% [-31.0%, -20.0%]
template source             180    91.1%    99.4%   100.0%       +8.9% [+5.0%, +13.3%]        +0.6% [+0.0%, +1.7%]
template first_untrusted    153    96.1%   100.0%    89.5%       -6.5% [-12.4%, -0.7%]      -10.5% [-15.7%, -5.9%]
template after_untrusted     69    31.9%    68.1%    97.1%     +65.2% [+53.6%, +76.8%]     +29.0% [+18.8%, +40.6%]
  full log      accuracy 87.5% [Wilson 85.7%, 89.2%] on 1332 questions
  search agent  accuracy 96.7% [Wilson 95.6%, 97.5%] on 1332 questions
  PROV graph    accuracy 93.5% [Wilson 92.0%, 94.7%] on 1332 questions


In [11]:
print("List questions with a non-empty gold set: completeness (recall) and precision")
print(f"{'':<16}{'n':>5}" + "".join(f"{c + ' rec':>14}{c + ' prec':>14}" for c in CONDS))
for L in list(LENGTHS) + ["all"]:
    qs = [q for q in QUESTIONS if q["kind"] == "list" and q["gold"] and (L == "all" or q["length"] == L)]
    if qs:
        print(f"{str(L) + ('-run logs' if L != 'all' else ''):<16}{len(qs):>5}" + "".join(
            f"{np.nanmean([SC[c][q['qid']]['rec'] for q in qs] + [np.nan]):>14.1%}{np.nanmean([SC[c][q['qid']]['prec'] for q in qs] + [np.nan]):>14.1%}"
            for c in CONDS))
print("\nTokens per answer (all calls of the condition), mean by log length")
print(f"{'':<16}" + "".join(f"{NAMES[c]:>14}" for c in CONDS))
for L in LENGTHS:
    qs = [q for q in QUESTIONS if q["length"] == L]
    print(f"{str(L) + '-run logs':<16}" + "".join(f"{tok(c, qs):>14,.0f}" for c in CONDS))
print(f"\nunparsed answers: " + ", ".join(f"{NAMES[c]} {sum(bool(RES[c][q['qid']]) and RES[c][q['qid']]['parsed'] is None for q in QUESTIONS)}" for c in CONDS))
print(f"graph queries that still failed after the repair: {sum(bool(G_STATE[q['qid']]['err']) for q in QUESTIONS)}; "
      f"repaired: {sum(G_STATE[q['qid']]['repaired'] for q in QUESTIONS)}; "
      f"empty result: {sum(bool(G_STATE[q['qid']]['out']) and G_STATE[q['qid']]['out'].endswith('(0 rows)') for q in QUESTIONS)}")

List questions with a non-empty gold set: completeness (recall) and precision
                    n      full rec     full prec    search rec   search prec     graph rec    graph prec
1-run logs        168         99.4%         99.4%        100.0%        100.0%         90.5%         97.1%
10-run logs       169         97.8%         95.0%         97.3%         99.5%         94.1%         97.4%
50-run logs       159         94.6%         95.6%         95.9%         99.8%         86.8%         96.7%
200-run logs      162         89.7%         93.0%         96.9%         98.4%         91.0%         98.7%
all               658         95.4%         95.8%         97.6%         99.4%         90.6%         97.5%

Tokens per answer (all calls of the condition), mean by log length
                      full log  search agent    PROV graph
1-run logs               1,357         2,553         1,614
10-run logs              7,267         3,639         1,614
50-run logs             35,283         4,

### One question, three answers

The first 50-run question on which the graph and at least one baseline disagree in correctness (an illustration,
not a typical case).

In [12]:
qs = [q for q in QUESTIONS if q["length"] == 50] or QUESTIONS
ex = next((q for q in qs if len({SC[c][q["qid"]]["acc"] for c in CONDS if not np.isnan(SC[c][q["qid"]]["acc"])}) > 1), qs[0])
print(f"[{ex['log']}, {ex['template']}] {ex['q']}\nGold: {ex['gold']}")
for c in CONDS:
    a = SC[c][ex["qid"]]["acc"]
    print(f"  {NAMES[c]:<13} {'not run' if np.isnan(a) else 'correct' if a else 'wrong  '}  {((RES[c][ex['qid']] or {}).get('parsed') or {}).get('answer')}")
print("\nSPARQL:\n" + G_STATE[ex["qid"]]["sparql"])
print("Result:\n" + "\n".join((G_STATE[ex["qid"]]["out"] or G_STATE[ex["qid"]]["err"] or "").splitlines()[:8]))

[injecagent-L50-00, cause] In run `inj-provenance-dh-agentdojo-113`, the agent called `TodoistSearchTasks` with the arguments {"keywords":"IT Audit","status":"completed"}. Which entities did the agent step that issued this call use? List their ids.
Gold: ['inj-provenance-dh-agentdojo-113.req']
  full log      correct  ['inj-provenance-dh-agentdojo-113.req']
  search agent  correct  ['inj-provenance-dh-agentdojo-113.req']
  PROV graph    wrong    []

SPARQL:
SELECT DISTINCT ?entityId WHERE {
  ?call a audit:ToolCall ;
        audit:id "inj-provenance-dh-agentdojo-113.c1.1" ;
        audit:tool "TodoistSearchTasks" ;
        audit:args ?args ;
        prov:wasInformedBy ?step .
  FILTER(CONTAINS(?args, "\"keywords\":\"IT Audit\""))
  FILTER(CONTAINS(?args, "\"status\":\"completed\"))
  ?step prov:used ?entity .
  ?entity audit:id ?entityId .
}
Result:
ParseException: Expected SelectQuery, found 'FILTER'  (at char 280), (line:8, col:3)


## What this run cost, and a forecast of the full run

The forecast multiplies the measured tokens per question of each stage by the full number of questions per log
length at the Batch price (half the standard price this test paid).

In [13]:
STAGES = ["nb24_fulllog"] + [f"nb24_search_step{i}" for i in range(1, MAX_TOOL_STEPS + 2)] + \
         ["nb24_graph_query", "nb24_graph_repair", "nb24_graph_answer"]
STAGES = [s for s in STAGES if os.path.exists(f"{llm.cache}/{s}.jsonl")]
LEDGER = llm.ledger(STAGES)
print(f"{'stage':<26}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for s in STAGES:
    a = LEDGER[s]
    print(f"{s:<26}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.2f}{a['batch_share']:>11.0%}")
print(f"{'TOTAL':<26}{sum(LEDGER[s]['calls'] for s in STAGES):>8,}{sum(LEDGER[s]['in'] for s in STAGES):>13,}"
      f"{sum(LEDGER[s]['out'] for s in STAGES):>12,}{sum(LEDGER[s]['usd'] for s in STAGES):>8.2f}")
if TEST:
    from smash_batch import PRICE
    full_n = {L: Q_PER_CELL * len(RUNS) for L in LENGTHS}
    est, calls = 0.0, 0
    for c, stages in (("full", ["nb24_fulllog"]), ("search", [s for s in STAGES if "search" in s]), ("graph", [s for s in STAGES if "graph" in s])):
        # condition (a) is forecast only for the share of questions whose log fits the context (as measured here)
        fit_share = {L: np.mean([FITS[q["log"]] for q in QUESTIONS if q["length"] == L]) if c == "full" else 1.0 for L in LENGTHS}
        per_q = {L: (tok(c, [q for q in QUESTIONS if q["length"] == L]) or 0) * fit_share[L] for L in LENGTHS}
        per_q = {L: 0.0 if np.isnan(v) else v for L, v in per_q.items()}
        # split tokens into input/output with this condition's measured ratio
        tin = sum(LEDGER[s]["in"] for s in stages); tout = sum(LEDGER[s]["out"] for s in stages)
        share_out = tout / max(tin + tout, 1)
        usd = sum(full_n[L] * per_q[L] * ((1 - share_out) * PRICE["batch"][0] + share_out * PRICE["batch"][1]) / 1e6 for L in LENGTHS)
        ncalls = sum(LEDGER[s]["calls"] for s in stages) / max(1, sum(bool(RES[c][q["qid"]]) for q in QUESTIONS)) * \
                 sum(full_n[L] * fit_share[L] for L in LENGTHS)
        est += usd; calls += ncalls
        print(f"forecast, full run, {NAMES[c]:<13}: ~{ncalls:,.0f} calls, ${usd:.2f} (Batch)")
    probe = sum(PROBE[l]["prompt_tokens"] or 0 for l in PROBE) / max(1, len(PROBE)) * 0.10 / 1e6 * \
            sum(N_LOGS[L] * len(RUNS) for L in LENGTHS if L >= 50)
    print(f"forecast, context probes (standard tier, {sum(N_LOGS[L] * len(RUNS) for L in LENGTHS if L >= 50)} logs): ${probe:.2f}")
    print(f"forecast, full run, total       : ~{calls:,.0f} calls, ${est + probe:.2f}; cost_forecast.py: 9,000 calls, $2.47")
print(f"context probes paid in this run (standard tier, outside the ledger): ${PROBE_USD:.2f}")

stage                        calls    input tok  output tok     USD  via Batch
nb24_fulllog                 1,332   59,554,104      30,355    2.99       100%
nb24_search_step1            1,332    1,134,117      41,530    0.07       100%
nb24_search_step2            1,332    2,224,654      29,453    0.12       100%
nb24_search_step3              357    1,037,510       7,887    0.05       100%
nb24_search_step4              122      541,682       2,564    0.03       100%
nb24_graph_query             1,332    1,181,736     145,155    0.10       100%
nb24_graph_repair                5        5,482         728    0.00       100%
nb24_graph_answer            1,332      796,548      27,302    0.05       100%
TOTAL                        7,144   66,475,833     284,974    3.40
context probes paid in this run (standard tier, outside the ledger): $0.12


## Limitations

- **Gold from the same record.** The gold answers are computed from the same events as both records, so the
  experiment measures how well each condition retrieves and aggregates what was recorded, not whether the record is
  complete or faithful (an agent could act outside its log).
- **Template questions in the graph's vocabulary.** The questions use the terms of the schema (step, used, trust,
  tool output). The same definitions are given to every condition in words, and the text log has the same fields, but
  free-form audit questions written by auditors could favour the text log (more natural language) or the graph
  (harder joins). A paraphrased subset would test this.
- **Truncated contents.** Requests, outputs and answers are cut to 400 characters in both records to keep a 50-run
  log near 30k tokens and a 200-run log near 110k–170k; questions about the cut text are not asked.
- **Our trust labels.** `untrusted` marks all InjecAgent tool outputs and the Aura outputs that contain ticket or comment
  records; another labelling changes templates 6–7.
- **Log lengths.** 200 runs are still below the 287K–617K-token traces of TRAIL; the trend over 1/10/50/200 runs is the
  evidence, not an absolute length at which the text log fails.
- **One configuration.** One model, one prompt per condition, at most 3 search rounds and one query repair; each
  configuration was run once.

**References.** Moreau & Missier (eds.), *PROV-O: The PROV Ontology*, W3C Recommendation, 2013 · Souza et al.,
*PROV-AGENT*, IEEE eScience 2025 · Deshpande et al., *TRAIL*, 2025 · Bertsch et al., *Oolong*, COLM 2026.

In [14]:
print("=" * 120)
print(f"RESULTS SUMMARY — Notebook 3.6: audit trail, {MODE} run, gpt-6-luna (no reasoning), {len(QUESTIONS)} questions "
      f"on nb 13/18/19 traces")
print("=" * 120)
print(f"{'subset':<24}{'n':>5}{'full log':>10}{'search':>10}{'graph':>10}{'graph - full [95% CI]':>28}{'graph - search [95% CI]':>28}"
      f"{'tokens/answer f/s/g':>24}")
ci = lambda t: "not runnable" if np.isnan(t[0]) else f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]"
for label in ["all questions"] + [f"{L}-run logs" for L in LENGTHS]:
    S = SUMMARY.get(label)
    if S:
        print(f"{label:<24}{S['n']:>5}" + "".join(f"{S['acc'][c]:>10.1%}" for c in CONDS) + f"{ci(S['g-full']):>28}{ci(S['g-search']):>28}"
              + f"{'/'.join(format(S['tokens'][c], ',.0f') for c in CONDS):>24}")
S = SUMMARY["all questions"]
verdict = {k: "not runnable" if np.isnan(S[k][0]) else "better" if S[k][1] > 0 else "worse" if S[k][2] < 0
           else "no resolvable difference" for k in ("g-full", "g-search")}
if S["n_full"] < S["n"]:
    print(f"(graph - full log is computed on the {S['n_full']} of {S['n']} questions whose log fits the context)")
print(f"\nVerdict (primary metric, accuracy, all questions): graph vs full log: {verdict['g-full']}; "
      f"graph vs search agent: {verdict['g-search']}")
print("\nLecture message (Task 3.6): an audit trail recorded as a PROV graph can be queried; whether queries answer audit"
      "\nquestions more accurately than reading or grepping the same log, and how that changes with log length, is what"
      "\nthe rows above say.")

RESULTS SUMMARY — Notebook 3.6: audit trail, batch run, gpt-6-luna (no reasoning), 1332 questions on nb 13/18/19 traces
subset                      n  full log    search     graph       graph - full [95% CI]     graph - search [95% CI]     tokens/answer f/s/g
all questions            1332     87.5%     96.7%     93.5%        +5.9% [+3.7%, +8.1%]        -3.2% [-4.9%, -1.7%]      44,733/3,768/1,619
1-run logs                333     93.7%     99.4%     94.6%        +0.9% [-2.7%, +4.5%]        -4.8% [-7.5%, -2.4%]       1,357/2,553/1,614
10-run logs               333     89.2%     96.1%     94.6%        +5.4% [+1.2%, +9.6%]        -1.5% [-4.8%, +1.8%]       7,267/3,639/1,614
50-run logs               333     86.5%     96.4%     91.0%        +4.5% [-0.6%, +9.3%]        -5.4% [-9.0%, -1.8%]      35,283/4,483/1,627
200-run logs              333     80.8%     94.9%     93.7%      +12.9% [+7.8%, +18.0%]        -1.2% [-4.5%, +2.1%]     135,025/4,398/1,623

Verdict (primary metric, accuracy, all 

## Amendment (2026-11-08): SQL over the event table

*Added 2026-11-08, after the confidence run, as a sensitivity analysis (`baselines.md`, item 10). Written down here
before this condition was run. The protocol cell above is unchanged and **the original primary result stands**:
the primary comparisons remain (c) − (b) and (c) − (a) as reported above.*

**Why.** The `grep` search agent beats the PROV graph by 3.2 points. A condition that also *generates a query*, but over
a relational table instead of a graph, separates "graphs" from "query generation" as the reason for that gap.

**System (d), SQL over the event table.**
- *Data:* the event list of each log (the same `LOGS[...]["runs"]` the text log and the PROV graph are built from) is
  written to an in-memory SQLite database: one table `events` (one row per line of the text log except `END RUN`, same
  fields), plus two lookup tables that mirror structures the graph already has as separate nodes/edges: `step_used`
  (the `prov:used` edges of a step) and `call_args` (the graph's argument nodes, same flattening, `arg_pairs`). A check
  recovers the event list unchanged from the database, as for the other two records.
- *Query:* one LLM call gets the schema (DDL with column comments), a few sample rows, the same `RECORD_DOC`, and the
  question, and writes one SQLite `SELECT` in a ```sql block. The sample rows come from one run of the same system that
  is **not** in the question's log, so they never contain the answer.
- *Execution* (copied from Notebook 3.1c): read-only (`PRAGMA query_only`, a gate that admits only one `SELECT`/`WITH`
  statement), a progress-handler time limit (60 s, the limit of condition (c)), at most 100 rows shown to the answering
  call.
- *Retry:* if the query raises an error or times out, **one** retry with the error message (as in Notebook 3.1c's retry,
  shortened to one round so that it matches the single repair of condition (c); no retry on an empty result, since 109
  questions have gold 0 or an empty set).
- *Answer:* a final call answers from the question, the query and its result only, with the same `ANSWER_RULES` and
  `Answer` schema as condition (c).
- Same model, temperature 0, new stages `nb24_sql_try1`, `nb24_sql_try2`, `nb24_sql_answer`.

**Metric.** The same exact-match accuracy (`score`) on the same 1,332 questions.

**Contrasts** (paired 95% bootstrap over questions, `paired_boot`): **graph − SQL** and **search agent − SQL**, over
all questions and by log length, system and template; SQL accuracy with a Wilson interval; tokens per question (all
calls of the condition). Secondary: SQL error rate before and after the retry.

**Reading rule** (fixed now): if graph − SQL is not resolvable from 0 while search − SQL is positive, the gap to the
search agent is about generating a query (and answering from its result), not about graphs. If graph − SQL is
positive, the graph form helps beyond query generation; if negative, the relational form is easier for this model.
The verdict line names both: the original result and this one.

**Differences to condition (c) to keep in mind.** (d) sees a few sample rows, (c) saw no sample triples (its prompt is
frozen); this favours (d). Notebook 3.6 is not part of the forest plot of 11–19, so no multiple-comparison correction
applies here.

In [15]:
import sqlite3

SQL_DDL = """CREATE TABLE events (            -- one row per event of the audit log, in log order
  seq INTEGER PRIMARY KEY,                -- position in the log: 1, 2, ...
  event_id TEXT UNIQUE,                   -- <run> (the run itself) | <run>.req | <run>.sK | <run>.cK.J | <run>.oK.J | <run>.ans
  event_type TEXT,                        -- Run | UserRequest | Step | ToolCall | ToolOutput | FinalAnswer
  run_id TEXT,                            -- the run the event belongs to
  system TEXT,                            -- aura | metaqa | injecagent
  agent TEXT,                             -- agent configuration of the run
  step_index INTEGER,                     -- Step only: K
  parent_id TEXT,                         -- Step: informed_by (previous step; NULL for step 1) | ToolCall: issued_by (step) | ToolOutput: generated_by (tool call) | FinalAnswer: generated_by (last step)
  tool TEXT,                              -- ToolCall only: tool name
  args TEXT,                              -- ToolCall only: JSON arguments
  trust TEXT,                             -- UserRequest and ToolOutput: trusted | untrusted
  content TEXT                            -- UserRequest, ToolOutput, FinalAnswer: text (truncated to 400 characters)
);
CREATE TABLE step_used (                  -- step USED entity (the user request or tool outputs)
  step_id TEXT,                           -- events.event_id of a Step
  entity_id TEXT                          -- events.event_id of a UserRequest or ToolOutput
);
CREATE TABLE call_args (                  -- one row per scalar argument, per list element, and per field 'name.field' of an object argument
  call_id TEXT,                           -- events.event_id of a ToolCall
  key TEXT,                               -- argument name
  value TEXT                              -- value as a string
);"""
COLS = ("seq", "event_id", "event_type", "run_id", "system", "agent", "step_index", "parent_id", "tool", "args", "trust", "content")

def event_rows(runs):
    ev, used, args = [], [], []
    def add(**f):
        ev.append(tuple([len(ev) + 1] + [f.get(c) for c in COLS[1:]]))
    for r in runs:
        base = {"run_id": r["id"], "system": r["system"], "agent": r["agent"]}
        add(event_id=r["id"], event_type="Run", **base)
        q = r["request"]
        add(event_id=q["id"], event_type="UserRequest", trust=q["trust"], content=q["text"], **base)
        for st in r["steps"]:
            sid = f"{r['id']}.s{st['k']}"
            add(event_id=sid, event_type="Step", step_index=st["k"],
                parent_id=f"{r['id']}.s{st['k'] - 1}" if st["k"] > 1 else None, **base)
            used += [(sid, u) for u in st["used"]]
            for c in st["calls"]:
                add(event_id=c["id"], event_type="ToolCall", parent_id=sid, tool=c["tool"], args=c["args"], **base)
                args += [(c["id"], k, v) for k, v in arg_pairs(c["args"])]
                if c["out"]:
                    o = c["out"]
                    add(event_id=o["id"], event_type="ToolOutput", parent_id=c["id"], trust=o["trust"], content=o["text"], **base)
        if r["answer"]:
            a = r["answer"]
            add(event_id=a["id"], event_type="FinalAnswer", parent_id=f"{r['id']}.s{a['step']}", content=a["text"], **base)
    return ev, used, args

def build_db(runs):
    con = sqlite3.connect(":memory:", check_same_thread=False)
    con.executescript(SQL_DDL)
    ev, used, args = event_rows(runs)
    con.executemany(f"INSERT INTO events VALUES ({','.join('?' * len(COLS))})", ev)
    con.executemany("INSERT INTO step_used VALUES (?, ?)", used)
    con.executemany("INSERT INTO call_args VALUES (?, ?, ?)", args)
    con.commit()
    con.execute("PRAGMA query_only = ON")                # read-only from here on
    return con

def runs_from_db(con):
    used = defaultdict(list)
    for s, e in con.execute("SELECT step_id, entity_id FROM step_used ORDER BY rowid"):
        used[s].append(e)
    cargs = defaultdict(list)
    for c, k, v in con.execute("SELECT call_id, key, value FROM call_args ORDER BY rowid"):
        cargs[c].append((k, v))
    runs = []
    for row in con.execute(f"SELECT {', '.join(COLS)} FROM events ORDER BY seq"):
        e = dict(zip(COLS, row))
        typ, i = e["event_type"], e["event_id"]
        if typ == "Run":
            r = {"id": e["run_id"], "system": e["system"], "agent": e["agent"], "steps": [], "answer": None}
            runs.append(r); steps, calls = {}, {}
        elif typ == "UserRequest":
            r["request"] = {"id": i, "text": e["content"], "trust": e["trust"]}
        elif typ == "Step":
            steps[i] = {"k": e["step_index"], "used": sorted(used[i]), "calls": []}
            r["steps"].append(steps[i])
        elif typ == "ToolCall":
            calls[i] = {"id": i, "tool": e["tool"], "args": e["args"], "out": None}
            steps[e["parent_id"]]["calls"].append(calls[i])
            assert cargs[i] == arg_pairs(e["args"]), i                  # the argument table matches the graph's arg nodes
        elif typ == "ToolOutput":
            calls[e["parent_id"]]["out"] = {"id": i, "text": e["content"], "trust": e["trust"]}
        else:
            r["answer"] = {"id": i, "step": steps[e["parent_id"]]["k"], "text": e["content"]}
    return runs

# sample rows: one run of the same system that is NOT in the log (first such run in id order), so they never hold the answer
def sample_rows(lid):
    log = LOGS[lid]
    inlog = {r["id"] for r in log["runs"]}
    r = next(x for x in sorted(RUNS[log["system"]], key=lambda x: x["id"]) if x["id"] not in inlog)
    ev, used, args = event_rows([r])
    cut = lambda v, n: "NULL" if v is None else (str(v) if len(str(v)) <= n else str(v)[:n] + "…")
    out = ["events (the rows of that run, at most 14; long texts shortened here):", " | ".join(COLS)]
    out += [" | ".join(cut(v, 60 if c in ("content", "args") else 80) for c, v in zip(COLS, row)) for row in ev[:14]]
    out += ["", "step_used (first rows):", "step_id | entity_id"] + [" | ".join(x) for x in used[:3]]
    out += ["", "call_args (first rows):", "call_id | key | value"] + [" | ".join(cut(v, 60) for v in x) for x in args[:3]]
    return "\n".join(out)

DB = {}
t0 = time.time()
for lid in USED_LOGS:
    DB[lid] = build_db(LOGS[lid]["runs"])
    assert canon(runs_from_db(DB[lid])) == canon(LOGS[lid]["runs"]), lid
print(f"{len(DB)} SQLite databases built in {time.time() - t0:.0f} s; every event list was recovered unchanged from the "
      f"database (same information as the text log and the PROV graph).")
for L in LENGTHS:
    ls = [l for l in USED_LOGS if LOGS[l]["length"] == L]
    n = lambda t: np.mean([DB[l].execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for l in ls])
    print(f"  {L:>3}-run logs: {n('events'):,.0f} events, {n('step_used'):,.0f} step_used, {n('call_args'):,.0f} call_args rows")
print("\nExample of the sample rows shown in the prompt:\n" + sample_rows(USED_LOGS[0]))

165 SQLite databases built in 0 s; every event list was recovered unchanged from the database (same information as the text log and the PROV graph).
    1-run logs: 11 events, 3 step_used, 11 call_args rows
   10-run logs: 105 events, 32 step_used, 60 call_args rows
   50-run logs: 539 events, 165 step_used, 431 call_args rows
  200-run logs: 2,114 events, 645 step_used, 1,486 call_args rows

Example of the sample rows shown in the prompt:
events (the rows of that run, at most 14; long texts shortened here):
seq | event_id | event_type | run_id | system | agent | step_index | parent_id | tool | args | trust | content
1 | aura-iq000 | Run | aura-iq000 | aura | aura-independent | NULL | NULL | NULL | NULL | NULL | NULL
2 | aura-iq000.req | UserRequest | aura-iq000 | aura | aura-independent | NULL | NULL | NULL | NULL | trusted | How many DELMIA team members did Carlos say would attend the…
3 | aura-iq000.s1 | Step | aura-iq000 | aura | aura-independent | 1 | NULL | NULL | NULL | NULL | N

In [16]:
SQL_TIMEOUT = 60                                   # seconds per query, the limit of condition (c)
SQL_Q_SYS = ("You audit the actions of LLM agents. Their runs are recorded as rows of a SQLite database (one row per "
             "event of the audit log). Write ONE SQLite SELECT query whose result answers the question (it will be run "
             "on the database, and the answer will be written from its result only). Return only the query in one "
             "```sql code block.\n\nSchema:\n" + SQL_DDL + "\n\n" + RECORD_DOC +
             "\nTip: return id strings; use COUNT(DISTINCT ...) for counts; use LIKE or instr(LOWER(...), ...) for text.")
SQL_A_SYS = ("You audit the actions of LLM agents. A SQL query was run on the SQLite database of the events of the agent "
             "runs; answer the question from the query result only.\n\n" + RECORD_DOC + "\n\n" + ANSWER_RULES)

def extract_sql(text):                             # as in Notebook 3.1c
    m = re.findall(r"```(?:sql|sqlite)?\s*(.*?)```", text or "", re.S | re.I)
    return (m[-1] if m else text or "").strip().rstrip(";").strip()

def run_sql(lid, sql, timeout=SQL_TIMEOUT, max_rows=100_000):
    # read-only gate and progress-handler time limit as in Notebook 3.1c; result rendered as condition (c) renders SPARQL results
    if not re.match(r"\s*(SELECT|WITH)\b", sql or "", re.I):
        return "READ-ONLY GATE: only a single SELECT query is allowed", None
    con, t0 = DB[lid], time.time()
    con.set_progress_handler(lambda: int(time.time() - t0 > timeout), 10_000)
    try:
        cur = con.execute(sql)
        rows = cur.fetchmany(max_rows)
        head = [d[0] for d in cur.description] if cur.description else []
        cell = lambda x: "" if x is None else clip(str(x), 200)
        body = [" | ".join(cell(x) for x in row) for row in rows[:MAX_ROWS]]
        more = f"\n({len(rows)} rows; first {MAX_ROWS} shown)" if len(rows) > MAX_ROWS else f"\n({len(rows)} rows)"
        return None, " | ".join(head) + "\n" + "\n".join(body) + more
    except Exception as e:
        if time.time() - t0 > timeout:
            return f"Timeout: the query ran longer than {timeout} s", None
        return f"{type(e).__name__}: {str(e)[:300]}", None
    finally:
        con.set_progress_handler(None, 0)

SAMPLES = {lid: sample_rows(lid) for lid in USED_LOGS}
def sql_msgs(q):
    return [{"role": "system", "content": SQL_Q_SYS},
            {"role": "user", "content": f"Sample rows (from one run of the same system that is not in this log):\n"
                                        f"{SAMPLES[q['log']]}\n\nThis log has {LOGS[q['log']]['length']} runs.\n\n"
                                        f"Question: {q['q']}"}]

res = llm.run("nb24_sql_try1", {q["qid"]: LLM.body(sql_msgs(q), max_tokens=800) for q in QUESTIONS})
S_STATE = {}
for q in QUESTIONS:
    r = res[q["qid"]]
    sql = extract_sql(r["text"])
    err, out = run_sql(q["log"], sql)
    S_STATE[q["qid"]] = {"sql": sql, "text": r["text"], "err": err, "err1": err, "out": out,
                         "tokens": r.get("in", 0) + r.get("out", 0), "retried": False}
bad = [q for q in QUESTIONS if S_STATE[q["qid"]]["err"]]
print(f"SQL queries: {len(QUESTIONS)} | raised an error or timed out: {len(bad)}")
if bad:
    reqs = {q["qid"]: LLM.body(sql_msgs(q) + [{"role": "assistant", "content": S_STATE[q["qid"]]["text"]},
                                              {"role": "user", "content": f"The query failed: {S_STATE[q['qid']]['err']}. "
                                                                          "Fix it and return the corrected query."}],
                               max_tokens=800) for q in bad}
    res = llm.run("nb24_sql_try2", reqs)
    for q in bad:
        st, r = S_STATE[q["qid"]], res[q["qid"]]
        st["sql"] = extract_sql(r["text"])
        st["err"], st["out"] = run_sql(q["log"], st["sql"])
        st["tokens"] += r.get("in", 0) + r.get("out", 0); st["retried"] = True
    print(f"after one retry: {sum(bool(S_STATE[q['qid']]['err']) for q in QUESTIONS)} queries still fail")
reqs = {}
for q in QUESTIONS:
    st = S_STATE[q["qid"]]
    result = st["out"] if st["err"] is None else f"(the query failed: {st['err']})"
    reqs[q["qid"]] = LLM.body([{"role": "system", "content": SQL_A_SYS},
                               {"role": "user", "content": f"Question: {q['q']}\n\nSQL query:\n{st['sql']}\n\nResult:\n{result}"}],
                              max_tokens=1500, schema=Answer)
res = llm.run("nb24_sql_answer", reqs, parse=Answer)
RES["sql"] = {}
for q in QUESTIONS:
    S_STATE[q["qid"]]["tokens"] += res[q["qid"]].get("in", 0) + res[q["qid"]].get("out", 0)
    RES["sql"][q["qid"]] = {"parsed": res[q["qid"]]["parsed"], "tokens": S_STATE[q["qid"]]["tokens"]}
print("SQL answers:", len(res), "| unparsed:", sum(r["parsed"] is None for r in res.values()))

[nb24_sql_try1] 1,332 new requests -> batch (0 cached)


[nb24_sql_try1] uploaded input file 1/1 (1,332 requests, 8 MB)


[nb24_sql_try1] started batch batch_6af0bdd3c174819081727856c7736a21


[nb24_sql_try1] 1 min: 1 running, 0 queued, 0 results cached


[nb24_sql_try1] 6 min: 1 running, 0 queued, 0 results cached


[nb24_sql_try1] 11 min: 1 running, 0 queued, 0 results cached


... (70 outputs left out)

[nb24_sql_answer] 147 min: 1 running, 0 queued, 0 results cached


[nb24_sql_answer] batch batch_6af0e9da59c08190951f1c6357b7f7c5 completed: 1,332 ok, 0 failed


[nb24_sql_answer] done in 149.1 min


SQL answers: 1332 | unparsed: 0


In [17]:
NAMES["sql"] = "SQL"
SC["sql"] = {q["qid"]: score(q, RES["sql"][q["qid"]]["parsed"]) for q in QUESTIONS}
SQL_SUMMARY = {}

def sql_table(label, qs):
    if not qs:
        return
    acc = {c: [SC[c][q["qid"]]["acc"] for q in qs] for c in ("search", "graph", "sql")}
    g, s = paired_boot(acc["sql"], acc["graph"]), paired_boot(acc["sql"], acc["search"])   # mean of (graph - SQL), (search - SQL)
    SQL_SUMMARY[label] = {"n": len(qs), "acc": {c: np.mean(v) for c, v in acc.items()}, "g-sql": g, "s-sql": s,
                          "tokens": {c: tok(c, qs) for c in ("search", "graph", "sql")}}
    S = SQL_SUMMARY[label]
    print(f"{label:<26}{len(qs):>5}" + "".join(f"{S['acc'][c]:>9.1%}" for c in ("search", "graph", "sql"))
          + "".join(f"{f'{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]':>28}" for t in (g, s))
          + f"{'/'.join(format(S['tokens'][c], ',.0f') for c in ('search', 'graph', 'sql')):>22}")

print(f"{'accuracy':<26}{'n':>5}{'search':>9}{'graph':>9}{'SQL':>9}{'graph - SQL [95% CI]':>28}{'search - SQL [95% CI]':>28}"
      f"{'tokens/q s/g/SQL':>22}")
sql_table("all questions", QUESTIONS)
for L in LENGTHS:
    sql_table(f"{L}-run logs", [q for q in QUESTIONS if q["length"] == L])
for s in RUNS:
    sql_table(f"system {s}", [q for q in QUESTIONS if q["system"] == s])
for t in TEMPLATES["aura"]:
    sql_table(f"template {t}", [q for q in QUESTIONS if q["template"] == t])
xs = [SC["sql"][q["qid"]]["acc"] for q in QUESTIONS]
lo, hi = wilson(sum(xs), len(xs))
print(f"\n  SQL accuracy {np.mean(xs):.1%} [Wilson {lo:.1%}, {hi:.1%}] on {len(xs)} questions")
qs = [q for q in QUESTIONS if q["kind"] == "list" and q["gold"]]
print(f"  list questions with a non-empty gold set ({len(qs)}): SQL completeness "
      f"{np.nanmean([SC['sql'][q['qid']]['rec'] for q in qs] + [np.nan]):.1%}, precision "
      f"{np.nanmean([SC['sql'][q['qid']]['prec'] for q in qs] + [np.nan]):.1%}")
print(f"  SQL errors: first query {sum(bool(S_STATE[q['qid']]['err1']) for q in QUESTIONS)} "
      f"({np.mean([bool(S_STATE[q['qid']]['err1']) for q in QUESTIONS]):.1%}), after the retry "
      f"{sum(bool(S_STATE[q['qid']]['err']) for q in QUESTIONS)}; empty results "
      f"{sum(bool(S_STATE[q['qid']]['out']) and S_STATE[q['qid']]['out'].endswith('(0 rows)') for q in QUESTIONS)}")

S = SQL_SUMMARY["all questions"]
word = lambda t: "better" if t[1] > 0 else "worse" if t[2] < 0 else "no resolvable difference"
print(f"\nVerdict (amendment, sensitivity analysis; the primary result above stands): graph vs SQL: {word(S['g-sql'])} "
      f"({S['g-sql'][0]:+.1%}); search agent vs SQL: {word(S['s-sql'])} ({S['s-sql'][0]:+.1%}).")

SQL_STAGES = [s for s in ("nb24_sql_try1", "nb24_sql_try2", "nb24_sql_answer") if os.path.exists(f"{llm.cache}/{s}.jsonl")]
L_SQL = llm.ledger(SQL_STAGES)
print(f"\n{'stage':<26}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for s in SQL_STAGES:
    a = L_SQL[s]
    print(f"{s:<26}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.2f}{a['batch_share']:>11.0%}")
print(f"{'TOTAL (SQL condition)':<26}{sum(L_SQL[s]['calls'] for s in SQL_STAGES):>8,}{sum(L_SQL[s]['in'] for s in SQL_STAGES):>13,}"
      f"{sum(L_SQL[s]['out'] for s in SQL_STAGES):>12,}{sum(L_SQL[s]['usd'] for s in SQL_STAGES):>8.2f}")

accuracy                      n   search    graph      SQL        graph - SQL [95% CI]       search - SQL [95% CI]      tokens/q s/g/SQL
all questions              1332    96.7%    93.5%    99.0%        -5.6% [-6.9%, -4.2%]        -2.3% [-3.4%, -1.3%]     3,768/1,619/2,132
1-run logs                  333    99.4%    94.6%    99.7%        -5.1% [-7.5%, -2.7%]        -0.3% [-1.5%, +0.6%]     2,553/1,614/2,114
10-run logs                 333    96.1%    94.6%    99.1%        -4.5% [-6.9%, -2.4%]        -3.0% [-5.4%, -0.9%]     3,639/1,614/2,104
50-run logs                 333    96.4%    91.0%    98.8%       -7.8% [-11.1%, -4.8%]        -2.4% [-4.8%, +0.0%]     4,483/1,627/2,136
200-run logs                333    94.9%    93.7%    98.5%        -4.8% [-7.5%, -2.4%]        -3.6% [-6.0%, -1.2%]     4,398/1,623/2,176
system injecagent           444    97.5%    96.2%    99.1%        -2.9% [-5.0%, -0.9%]        -1.6% [-3.4%, +0.0%]     3,007/1,629/2,125
system aura                 444    92.8% 

system metaqa               444    99.8%    95.7%    99.1%        -3.4% [-5.4%, -1.6%]        +0.7% [-0.2%, +1.6%]     4,189/1,624/2,168
template count_runs         229    93.9%    98.7%   100.0%        -1.3% [-3.1%, +0.0%]        -6.1% [-9.6%, -3.1%]     4,923/1,578/2,058
template list_runs          232    99.6%    99.6%   100.0%        -0.4% [-1.3%, +0.0%]        -0.4% [-1.3%, +0.0%]     4,652/1,632/2,117
template count_calls        224    98.7%   100.0%   100.0%        +0.0% [+0.0%, +0.0%]        -1.3% [-3.1%, +0.0%]     2,563/1,485/1,999
template cause              245    98.8%    73.5%    97.1%     -23.7% [-29.4%, -18.4%]        +1.6% [-0.8%, +4.1%]     2,737/1,742/2,311
template source             180    99.4%   100.0%    97.8%        +2.2% [+0.6%, +5.0%]        +1.7% [-0.6%, +4.4%]     2,837/1,663/2,085
template first_untrusted    153   100.0%    89.5%   100.0%      -10.5% [-15.7%, -5.9%]        +0.0% [+0.0%, +0.0%]     2,474/1,571/2,207


template after_untrusted     69    68.1%    97.1%    97.1%        +0.0% [+0.0%, +0.0%]     -29.0% [-40.6%, -18.8%]     9,837/1,712/2,187

  SQL accuracy 99.0% [Wilson 98.3%, 99.4%] on 1332 questions
  list questions with a non-empty gold set (658): SQL completeness 98.7%, precision 99.7%
  SQL errors: first query 0 (0.0%), after the retry 0; empty results 76

Verdict (amendment, sensitivity analysis; the primary result above stands): graph vs SQL: worse (-5.6%); search agent vs SQL: worse (-2.3%).

stage                        calls    input tok  output tok     USD  via Batch
nb24_sql_try1                1,332    1,894,476     119,875    0.12       100%
nb24_sql_answer              1,332      798,187      27,803    0.05       100%
TOTAL (SQL condition)        2,664    2,692,663     147,678    0.17


### Amendment results and verdict (SQL over the event table)

- **Overall:** SQL accuracy 99.0% [Wilson 98.3%, 99.4%] on 1,332 questions, against 96.7% for the `grep` search agent
  and 93.5% for the provenance graph. Graph − SQL −5.6 points [−6.9, −4.2]; search agent − SQL −2.3 [−3.4, −1.3].
  No SQL query failed, before or after the retry; 76 results were empty (109 questions have an empty or zero gold
  answer).
- **By log length:** SQL stays at 98.5–99.7% from 1-run to 200-run logs; graph − SQL is between −4.5 and −7.8 at every
  length.
- **By template:** the graph's largest losses are on `cause` (−23.7) and `first_untrusted` (−10.5); it is ahead only on
  `source` (+2.2 [+0.6, +5.0]). The search agent's weak point, `after_untrusted` (68.1%), is solved by SQL (97.1%), as
  by the graph.
- **Tokens per question:** search 3,768 / graph 1,619 / SQL 2,132.
- **Caveat stated in advance:** the SQL prompt shows sample rows (from a run of the same system that is not in the
  question's log), which the frozen graph prompt did not get. This favours SQL somewhat.
- **Verdict** (sensitivity analysis; the primary result stands): graph better than the full log and worse than the
  search agent (primary); **graph worse than SQL over the same events (−5.6)**; search agent worse than SQL (−2.3).
  The audit trail is best kept as a typed event table queried with SQL; the provenance graph adds nothing here.
  Cost of the amendment: $0.17 at Batch prices.